# Reducción de Dimensionalidad — PCA

_Componentes principales, varianza explicada y visualización de clusters K-Means + PCA_

**Módulo 2 — Aprendizaje No Supervisado** | DSRP Machine Learning Engineering
**Profesor:** Miguel Arquez

![Aprendizaje No Supervisado](assets/header.png)

## 1. ¿Por qué reducir dimensionalidad?

Cuando $p$ es grande (decenas o cientos de variables) aparecen problemas:

- **La maldición de la dimensión**: las distancias entre puntos se vuelven similares y los algoritmos basados en distancia (K-Means, KNN) pierden poder discriminativo.
- **Redundancia**: muchas variables suelen estar correlacionadas y aportan información repetida.
- **Visualización**: no podemos graficar más de 2-3 dimensiones.
- **Costo computacional** y **ruido**.

**Idea de PCA:** encontrar un número pequeño de **nuevos ejes** (combinaciones lineales de las variables originales) que **capturen la mayor varianza posible** de los datos.

## 2. Intuición geométrica

Imagina una nube de puntos alargada en 2D. PCA encuentra:
- **PC1** — la dirección a lo largo de la cual los datos varían más (eje "largo" de la elipse).
- **PC2** — la dirección **ortogonal** a PC1 que captura la varianza restante.

Si proyectamos los datos sobre PC1 perdemos muy poca información (los puntos están "casi" sobre esa línea).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

rng = np.random.default_rng(0)
mean = [0, 0]
cov = [[3, 1.6], [1.6, 1]]
X = rng.multivariate_normal(mean, cov, size=400)

pca = PCA(n_components=2).fit(X)
comps = pca.components_
expl = pca.explained_variance_

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(X[:, 0], X[:, 1], alpha=0.4, s=15)
for i, (vec, var) in enumerate(zip(comps, expl)):
    v = vec * np.sqrt(var) * 2
    ax.annotate('', xy=v, xytext=(0, 0),
                arrowprops={'arrowstyle': '->', 'lw': 3, 'color': 'red'})
    ax.text(v[0]*1.1, v[1]*1.1, f'PC{i+1}', fontsize=14,
            color='red', fontweight='bold')
ax.set_aspect('equal'); ax.grid(True)
ax.set_title('PCA: las flechas son los ejes de máxima varianza')
plt.show()

## 3. Formulación matemática

Sea $X \in \mathbb{R}^{n \times p}$ centrada (media cero por columna). La matriz de covarianza es:

$$
\Sigma = \frac{1}{n-1} X^\top X
$$

Los **componentes principales** son los autovectores de $\Sigma$ ordenados por sus autovalores $\lambda_1 \ge \lambda_2 \ge \dots \ge \lambda_p$:

$$
\Sigma v_k = \lambda_k v_k
$$

- $v_k$ es la **dirección** del $k$-ésimo componente.
- $\lambda_k$ es la **varianza** explicada por ese componente.

Las nuevas coordenadas (scores) se obtienen proyectando:

$$
Z = X V_k \in \mathbb{R}^{n \times k}
$$

donde $V_k$ contiene los $k$ primeros autovectores. En la práctica, scikit-learn lo calcula vía **SVD** sobre $X$, que es numéricamente más estable que diagonalizar $\Sigma$.

## 4. Varianza explicada — ¿cuántos componentes elegir?

Cada componente captura una fracción de la varianza total:

$$
\text{ratio}_k = \frac{\lambda_k}{\sum_{j=1}^{p} \lambda_j}
$$

Tres heurísticas comunes:

1. **Umbral acumulado**: quedarnos con los $k$ primeros componentes que acumulen, p. ej., el 80–95% de la varianza total.
2. **Codo del scree plot**: graficar $\lambda_k$ vs $k$ y cortar donde se aplana.
3. **Regla de Kaiser**: conservar solo componentes con $\lambda_k > 1$ (cuando los datos están estandarizados — significa "explica más que una variable promedio").

## 5. Pre-requisitos importantes

- **Centrado** y **escalado** de las variables. Sin esto, PCA se enfoca en los ejes con mayor magnitud (no con mayor _información_). `StandardScaler` antes de PCA es la regla.
- **Linealidad**: PCA solo encuentra estructuras lineales. Para no lineales se usa **Kernel PCA**, **t-SNE** o **UMAP** (estos dos últimos pensados para visualización, no como features).
- **Interpretabilidad**: los componentes son combinaciones lineales de las variables originales — útil pero no siempre fácil de explicar al negocio.

## 6. Caso práctico — PCA sobre Telco Customer Churn

Vamos a:
1. Aplicar PCA al dataset de Telco con muchas variables (numéricas + dummies).
2. Mirar cuántos componentes acumulan ≥80% de la varianza.
3. Inspeccionar qué variables pesan en cada componente (loadings).
4. Combinar **K-Means + PCA** para visualizar los segmentos en 2D — el caso más útil de PCA en clustering.

In [ ]:
from pathlib import Path
import pandas as pd

DATA = Path('../data/WA_Fn-UseC_-Telco-Customer-Churn.csv')
if not DATA.exists():
    raise FileNotFoundError(
        f'No se encontró {DATA}. Descarga el dataset desde '
        'https://www.kaggle.com/datasets/blastchar/telco-customer-churn '
        'y colócalo en data/ (ver README.md).'
    )

df = pd.read_csv(DATA)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df = df.dropna(subset=['TotalCharges']).reset_index(drop=True)
print('Shape:', df.shape)
df.head()

In [ ]:
import seaborn as sns
from sklearn.preprocessing import StandardScaler

sns.set_theme(style='whitegrid')

num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges', 'SeniorCitizen']
cat_cols = ['gender', 'Partner', 'Dependents', 'PhoneService',
            'InternetService', 'Contract', 'PaperlessBilling', 'PaymentMethod']

X = pd.get_dummies(df[num_cols + cat_cols], columns=cat_cols, drop_first=True)
X = X.astype(float)
print('Dimensión original:', X.shape)

scaler = StandardScaler().fit(X)
X_s = scaler.transform(X)

In [ ]:
from sklearn.decomposition import PCA

# Ajustamos PCA con todos los componentes posibles para inspeccionar la varianza
pca_full = PCA().fit(X_s)
ratio = pca_full.explained_variance_ratio_
acum = np.cumsum(ratio)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].bar(range(1, len(ratio) + 1), ratio, color='steelblue')
axes[0].set_xlabel('Componente'); axes[0].set_ylabel('Varianza explicada')
axes[0].set_title('Scree plot')

axes[1].plot(range(1, len(acum) + 1), acum, marker='o')
axes[1].axhline(0.8, color='red', ls='--', label='80%')
axes[1].axhline(0.9, color='green', ls='--', label='90%')
axes[1].set_xlabel('Componente'); axes[1].set_ylabel('Varianza acumulada')
axes[1].set_title('Varianza acumulada')
axes[1].legend()
plt.tight_layout(); plt.show()

k80 = int(np.searchsorted(acum, 0.80) + 1)
k90 = int(np.searchsorted(acum, 0.90) + 1)
print(f'Componentes para ≥80% varianza: {k80}')
print(f'Componentes para ≥90% varianza: {k90}')

In [ ]:
# --- Loadings: qué variables pesan en cada componente ---
n_show = 4
loadings = pd.DataFrame(
    pca_full.components_[:n_show].T,
    index=X.columns,
    columns=[f'PC{i+1}' for i in range(n_show)],
)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(loadings, annot=True, fmt='.2f', center=0,
            cmap='RdBu_r', cbar_kws={'label': 'loading'}, ax=ax)
ax.set_title(f'Loadings de los primeros {n_show} componentes')
plt.tight_layout(); plt.show()

**Cómo leer los loadings:**
- Un valor positivo grande (azul) → la variable empuja la observación hacia el lado positivo del componente.
- Un valor negativo grande (rojo) → empuja al lado negativo.
- Valores cercanos a 0 → esa variable no contribuye al componente.

Esto nos permite ponerle un nombre a cada PC (p. ej., "PC1 = antigüedad y gasto acumulado").

## 7. Aplicación: K-Means + PCA para visualizar clusters

Esta es **la combinación más usada** de PCA en la práctica del clustering: no porque PCA mejore al K-Means (a veces sí, a veces no), sino porque nos permite **mirar** los clusters en un plano 2D y validarlos visualmente.

Hay dos formas de combinarlos:

| Estrategia | Cómo |
|---|---|
| **Clusterizar en alta dimensión, visualizar con PCA** | Entrenar K-Means sobre $X$ escalado y proyectar las etiquetas sobre los 2 primeros PCs |
| **Reducir primero con PCA y clusterizar en el espacio reducido** | Aplicar PCA a $k$ componentes y luego K-Means sobre $Z$ — útil cuando hay mucho ruido o redundancia |

Mostramos ambas.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

K = 4

# --- Estrategia 1: clusterizar en alta dimensión, visualizar con PCA(2) ---
km_full = KMeans(n_clusters=K, n_init=20, random_state=42).fit(X_s)
sil_full = silhouette_score(X_s, km_full.labels_)

pca2 = PCA(n_components=2).fit(X_s)
Z2 = pca2.transform(X_s)

# --- Estrategia 2: PCA primero, K-Means después ---
pcaK = PCA(n_components=k80).fit(X_s)   # k80 explica ≥80% de la varianza
Z_red = pcaK.transform(X_s)
km_red = KMeans(n_clusters=K, n_init=20, random_state=42).fit(Z_red)
sil_red = silhouette_score(Z_red, km_red.labels_)

print(f'Silhouette en alta dimensión       : {sil_full:.3f}')
print(f'Silhouette tras reducir con PCA({k80}): {sil_red:.3f}')

In [ ]:
# --- Visualización 2D ---
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Centroides reproyectados sobre PC1-PC2
centros2_full = pca2.transform(km_full.cluster_centers_)
axes[0].scatter(Z2[:, 0], Z2[:, 1], c=km_full.labels_,
                cmap='tab10', s=10, alpha=0.6)
axes[0].scatter(centros2_full[:, 0], centros2_full[:, 1],
                marker='X', c='black', s=200, label='centroides')
axes[0].set_xlabel(f'PC1 ({pca2.explained_variance_ratio_[0]*100:.1f}%)')
axes[0].set_ylabel(f'PC2 ({pca2.explained_variance_ratio_[1]*100:.1f}%)')
axes[0].set_title('K-Means en alta dim — proyectado en PCA(2)')
axes[0].legend()

# Para la estrategia 2, los centroides ya viven en el espacio PCA reducido
centros2_red = km_red.cluster_centers_[:, :2]
axes[1].scatter(Z_red[:, 0], Z_red[:, 1], c=km_red.labels_,
                cmap='tab10', s=10, alpha=0.6)
axes[1].scatter(centros2_red[:, 0], centros2_red[:, 1],
                marker='X', c='black', s=200, label='centroides')
axes[1].set_xlabel('PC1'); axes[1].set_ylabel('PC2')
axes[1].set_title(f'K-Means tras PCA({k80})')
axes[1].legend()
plt.tight_layout(); plt.show()

**Lectura de la visualización:**
- Si en el scatter PC1×PC2 vemos **manchas separadas** del color de cada cluster, la estructura es clara y robusta.
- Si los colores se entremezclan, el clustering puede no ser el adecuado para los datos — o los 2 primeros PCs no capturan suficiente varianza para ser representativos.
- **Cuidado**: 2 componentes pueden explicar solo, p. ej., 30% de la varianza. Que se vean mezclados en PC1×PC2 no significa necesariamente que los clusters estén mal.

In [ ]:
# --- Validación externa con Churn ---
df['cluster_pca'] = km_red.labels_
df['Churn_bin'] = (df['Churn'] == 'Yes').astype(int)

resumen = df.groupby('cluster_pca').agg(
    n_clientes=('Churn_bin', 'size'),
    tasa_churn=('Churn_bin', 'mean'),
    tenure_medio=('tenure', 'mean'),
    cargo_mensual_medio=('MonthlyCharges', 'mean'),
).round(3)
resumen

## 8. Cuándo NO usar PCA

- Cuando las variables originales **ya son interpretables** y el negocio quiere razonar sobre ellas (PCA crea ejes abstractos).
- Cuando la estructura es **no lineal** — t-SNE / UMAP / Kernel PCA suelen visualizarla mejor.
- Cuando $p$ es **muy pequeño** (p. ej., 3–5 variables): no hay nada que reducir.
- Cuando los **outliers** dominan: la varianza está inflada por unos pocos puntos extremos.

## 9. Referencias

- Pearson, K. (1901). *On lines and planes of closest fit to systems of points in space*.
- Hotelling, H. (1933). *Analysis of a complex of statistical variables into principal components*.
- ISLR cap. 12.2 — *Principal Components Analysis*.
- scikit-learn: https://scikit-learn.org/stable/modules/decomposition.html#pca
- Jolliffe, I. T. (2002). *Principal Component Analysis*.

## Predicción sobre datos nuevos — uso del pipeline en producción

PCA es **transformador**, no predictor — pero es una pieza fundamental del pipeline. La regla de oro: el `scaler`, el `pca` y el `kmeans` deben persistirse **juntos** y aplicarse en el mismo orden a los datos nuevos.

In [ ]:
import joblib

scaler_final = StandardScaler().fit(X)
X_s_final = scaler_final.transform(X)

pca_final = PCA(n_components=k80).fit(X_s_final)
km_final = KMeans(n_clusters=K, n_init=20, random_state=42).fit(
    pca_final.transform(X_s_final)
)

bundle = {
    'scaler':   scaler_final,
    'pca':      pca_final,
    'kmeans':   km_final,
    'columnas': X.columns.tolist(),
    'num_cols': num_cols,
    'cat_cols': cat_cols,
    'K':        K,
}
joblib.dump(bundle, 'pipeline_telco_pca_kmeans.pkl')
print('Pipeline guardado: pipeline_telco_pca_kmeans.pkl')

### Inferencia individual — un cliente nuevo pasando por todo el pipeline

In [ ]:
nuevo_cliente_raw = pd.DataFrame([{
    'tenure':           5,
    'MonthlyCharges':   85.0,
    'TotalCharges':     425.0,
    'SeniorCitizen':    0,
    'gender':           'Female',
    'Partner':          'No',
    'Dependents':       'No',
    'PhoneService':     'Yes',
    'InternetService':  'Fiber optic',
    'Contract':         'Month-to-month',
    'PaperlessBilling': 'Yes',
    'PaymentMethod':    'Electronic check',
}])

# Mismo encoding y mismo orden de columnas que en entrenamiento
nuevo = pd.get_dummies(nuevo_cliente_raw, columns=cat_cols, drop_first=True)
nuevo = nuevo.reindex(columns=X.columns, fill_value=0).astype(float)

z = pca_final.transform(scaler_final.transform(nuevo))
cluster_id = km_final.predict(z)[0]
print(f'Coordenadas en PCA: PC1={z[0, 0]:.2f}, PC2={z[0, 1]:.2f}')
print(f'Cluster asignado: {cluster_id}')

**Lecciones para producción:**
- El número de componentes (`k80`) es un hiperparámetro: documenta en qué porcentaje de varianza acumulada lo elegiste.
- Si las variables originales cambian (nueva campaña, nuevo producto) hay que **reentrenar PCA**, no solo el K-Means.
- Para visualizaciones rápidas en dashboards, persistir el `pca` con `n_components=2` es muy útil — permite ubicar nuevos clientes en el mismo mapa que se usó para presentar la segmentación al negocio.